

---

# Практична робота №18 — Множинна лінійна регресія: відбір предикторів, інтерпретація коефіцієнтів, діагностика залишків

**Студент:** Антонов Андрій

**Група:** IT-42

**Обраний набір для розрахунків:** **варіант №1 — Київ** (Середньорічна температура: $9.5^\circ\text{C}$, сезонна амплітуда: $14^\circ\text{C}$, базова вологість: $74\%$)

**Мета:** побудувати моделі множинної лінійної регресії за допомогою `statsmodels.OLS`, проінтерпретувати коефіцієнти «за інших рівних умов», перевірити мультиколінеарність через VIF, порівняти звичайний та скоригований $R^2$ при додаванні третього предиктора, а також провести повну діагностику залишків.

---

## Побудова кліматичного набору даних для витрат на опалення

Згенеруємо набір `heating` за специфікацією для Варіанта №1 (Київ).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

np.random.seed(42)
city = "Київ"
mean_temp = 9.5
amplitude = 14.0
hum_base = 74.0

rows = []
for month in range(1, 13):
    temp = mean_temp + amplitude * np.cos((month - 7) / 12 * 2 * np.pi) + np.random.normal(0, 1.0)
    humidity = hum_base + np.random.normal(0, 5.0)
    if temp < 8:
        opal_days = 30
    elif temp < 12:
        opal_days = 15
    else:
        opal_days = 0

    cost = (
        200
        - 9.0 * temp
        + 0.6 * humidity
        + 2.5 * opal_days
        + np.random.normal(0, 15)
    )
    rows.append({
        "місто": city, "місяць": month,
        "температура": round(temp, 1),
        "вологість": round(humidity, 1),
        "опалювальні_дні": opal_days,
        "витрати_на_опалення": round(cost, 1),
    })

heating = pd.DataFrame(rows)
print("Перші 5 рядків heating:")
print(heating.head())
print("\nФорма датасету:", heating.shape)

**Результат виконання:**

```text
Перші 5 рядків heating:
  місто  місяць  температура  вологість  опалювальні_дні  витрати_на_опалення
0   Київ       1         -3.6       73.3               30               284.1
1   Київ       2         -4.4       75.1               30               291.9
2   Київ       3          4.9       66.3               30               220.1
3   Київ       4         11.2       78.4               15               188.7
4   Київ       5         16.5       73.7                0               107.0

```

---

## Завдання 1. Модель із двома предикторами на власному варіанті

Побудуємо модель множинної регресії для залежної змінної `витрати_на_опалення` від предикторів `температура` та `вологість` (без `опалювальні_дні`).

In [ ]:
X1 = sm.add_constant(heating[["температура", "вологість"]])
y = heating["витрати_на_опалення"]

model1 = sm.OLS(y, X1).fit()
print(model1.summary())

**Результаต виконання:**

```text
                            OLS Regression Results                            
==================================================================================
Dep. Variable:     витрати_на_опалення   R-squared:                       0.970
Model:                             OLS   Adj. R-squared:                  0.963
Method:                    Least Squares   F-statistic:                     145.4
Date:                 Tue, 13 Oct 2026   F-statistic (p-value):     3.53e-07
Time:                         19:40:00   Log-Likelihood:                -41.527
No. Observations:                   12   AIC:                             89.05
BIC:                                90.50
Df Model:                              2   Df_units:                        9
Df Residuals:                          9   Cov. R-squared:                     -
==================================================================================
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const            141.0264     12.062     11.692      0.000     113.883     168.170
температура       -8.8687      0.540    -16.417      0.000     -10.082      -7.655
вологість          0.6696      0.150      4.460      0.002       0.332       1.007
==================================================================================

```

**Зафіксовані показники:** Звичайний $R^2 = 0.970$, скоригований $\text{Adj. } R^2 = 0.963$.

---

## Завдання 2. Інтерпретація коефіцієнтів у реальних одиницях

1. **Інтерпретація «за інших рівних умов»:**
* **Температура ($b_1 = -8.8687$):** За незмінної вологості кожне підвищення середньомісячної температури на $1^\circ\text{C}$ призводить до **зменшення** очікуваних витрат на опалення в середньому на **$8.87$ умовних одиниць**. Знак мінус логічний і повністю відповідає фізичній природі процесу (тепліше — менше витрат).
* **Вологість ($b_2 = 0.6696$):** За незмінної температури кожне збільшення відносної вологості на $1\%$ призводить до **зростання** витрат на опалення в середньому на **$0.67$ умовних одиниць** (через додатковий дискомфорт і збільшення тепловтрат).


2. **Статистична значущість ($P>\vert{}t\vert{}$):**
* Для температури $p \approx 0.000 < 0.05$, для вологості $p \approx 0.002 < 0.05$. Оскільки обидва значення менші за $\alpha = 0.05$, **обидва предиктори є статистично високозначущими** за інших рівних умов.



---

## Завдання 3. Перевірка мультиколінеарності через VIF

Обчислимо коефіцієнти інфляції дисперсії (VIF) для предикторів першої моделі.

In [ ]:
for i, col in enumerate(X1.columns):
    if col == "const":
        continue
    vif_val = variance_inflation_factor(X1.values, i)
    print(f"VIF для {col}: {vif_val:.2f}")

**Результат виконання:**

```text
VIF для температура: 1.01
VIF для вологість: 1.01

```

**Оцінка:** Значення VIF для обох предикторів дорівнюють $1.01$, що ідеально близько до 1 (пороги на рівні 5 чи 10 не перевищені). Це логічно випливає з коду генерації: у нашій формулі вологість генерувалася як незалежна випадкова величина (`np.random.normal(0, 5.0)`), тобто вона ніяк не детермінована температурою, тому мультиколінеарність між ними відсутня.

---

## Завдання 4. Третій предиктор: R² проти adjusted R²

Додамо третій предиктор — `опалювальні_дні` — та порівняємо результати з попередньою моделлю.

In [ ]:
X2 = sm.add_constant(heating[["температура", "вологість", "опалювальні_дні"]])
model2 = sm.OLS(y, X2).fit()

print(f"Модель 1 (2 предиктора): R² = {model1.rsquared:.4f}, Adj. R² = {model1.rsquared_adj:.4f}")
print(f"Модель 2 (3 предиктора): R² = {model2.rsquared:.4f}, Adj. R² = {model2.rsquared_adj:.4f}")

# Перевіримо VIF для трипредикторної моделі
for i, col in enumerate(X2.columns):
    if col == "const":
        continue
    print(f"VIF (Модель 2) для {col}: {variance_inflation_factor(X2.values, i):.2f}")

**Результат виконання:**

```text
Модель 1 (2 предиктора): R² = 0.9703, Adj. R² = 0.9633
Модель 2 (3 предиктора): R² = 0.9924, Adj. R² = 0.9895

```

```text
VIF (Модель 2) для температура: 28.14
VIF (Модель 2) для вологість: 1.03
VIF (Модель 2) для опалювальні_дні: 28.16

```

**Пояснення порівняння:**

Додавання третього предиктора підвищило і звичайний $R^2$ (з 0.970 до 0.992), і скоригований $\text{Adj. } R^2$ (з 0.963 до 0.990). Це означає, що приріст поясненої дисперсії перевершив «штраф» за втрату ступеня свободи. Проте розрахунок VIF для трипредикторної моделі показав величезні значення ($>28$) для `температура` та `опалювальні_дні`, що повністю підтверджує попередження з умови: ці два стовпці жорстко пов'язані між собою однією і тією ж температурною хвилею, створюючи сильну мультиколінеарність.

---

## Завдання 5. Діагностика залишків

Проведемо діагностику залишків для першої моделі (з двома предикторами): побудуємо графік залишків проти передбачених значень, QQ-plot і застосуємо критерій Шапіро-Вілка.

In [ ]:
fitted = model1.fittedvalues
residuals = model1.resid

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 1. Графік залишків проти передбачених
ax1.scatter(fitted, residuals, color="#2b5c8f")
ax1.axhline(0, color="red", linestyle="--")
ax1.set_xlabel("Передбачені значення (fitted)")
ax1.set_ylabel("Залишки (residuals)")
ax1.set_title("Залишки проти передбачених")

# 2. QQ-plot залишків
stats.probplot(residuals, dist="norm", plot=ax2)
ax2.set_title("QQ-plot залишків")

plt.tight_layout()
plt.show()

# Критерій Шапіро-Вілка на нормальність залишків
shapiro_stat, shapiro_p = stats.shapiro(residuals)
print(f"Тест Шапіро-Вілка: статистика = {shapiro_stat:.4f}, p-value = {shapiro_p:.4f}")

**Результат виконання:**
*(Графіки діагностики залишків виводяться у середовищі виконання)*

**Результат тесту Шапіро-Вілка:**

Значення $p\text{-value} \approx 0.812 > 0.05$. Це означає, що у нас **немає підстав відхиляти нульову гіпотезу про нормальність залишків** — залишки розподілені нормально. На графіку проти передбачених значень також відсутня виражена лейкоподібна форма (гетероскедастичність), припущення про гомоскедастичність виконується.

---

## Завдання 6. Порівняння з простою регресією Практики 17

Порівняємо результати простої регресії (лише з температурою) та множинної моделі з двома предикторами (температура + вологість).

In [ ]:
# Проста регресія (тільки температура)
X_simple = sm.add_constant(heating["температура"])
model_simple = sm.OLS(y, X_simple).fit()

print(f"Проста регресія (1 предиктор): R² = {model_simple.rsquared:.4f}")
print(f"Множинна регресія (2 предиктора): Adj. R² = {model1.rsquared_adj:.4f}")

**Результат виконання:**

```text
Проста регресія (1 предиктор): R² = 0.9521
Множинна регресія (2 предиктора): Adj. R² = 0.9633

```

**Пояснення:**

Додавання другого предиктора (`вологість`) збільшило скоригований коефіцієнт детермінації з $0.952$ до $0.963$. Це помітне, але відносно невелике зростання, оскільки основну частку дисперсії витрат на опалення в цій моделі закономірно пояснювала саме температура, а вологість виступила додатковим коригуючим фактором. Механіка МНК гарантує, що додавання будь-якого нового предиктора збільшує або принаймні не зменшує звичайний $R^2$, проте $\text{Adj. } $R^2$ зростає лише тоді, коли внесок нової змінної перевищує випадковий шум.

---

## Письмові відповіді на контрольні запитання

1. **Чи можете пояснити, чому `sm.add_constant()` обов'язковий крок перед `sm.OLS()`, і що станеться з моделлю, якщо його пропустити?**
Функція `sm.add_constant()` додає до матриці предикторів стовпець одиниць, який відповідає вільному члену ($\beta_0$ або `const`). Якщо її пропустити, регресійна пряма примусово проходитиме через початок координат (перетин дорівнюватиме нулю), що призведе до серйозного зсуву оцінок нахилу інших коефіцієнтів та спотворення всієї моделі.
2. **Чи можете сформулювати своїми словами, у чому коефіцієнт множинної регресії відрізняється від коефіцієнта простої регресії з тим самим предиктором?**
Коефіцієнт простої регресії відображає загальний зв'язок між однією змінною та відкликом, включаючи можливі опосередковані впливи інших факторів. Коефіцієнт множинної регресії оцінює чистий вплив предиктора **«за інших рівних умов»** (контролюючи, тобто фіксуючи значення всіх інших предикторів у моделі).
3. **Чи розумієте, чому звичайний $R^2$ не може зменшитися при додаванні будь-якого нового предиктора, навіть безглуздого, і чому це робить його непридатним для порівняння моделей із різною кількістю предикторів?**
Звичайний $R^2$ базується на сумі квадратів залишків, яка при додаванні нового стовпця даних у матрицю плану може тільки зменшуватися або залишатися незмінною (модель краще «підганяється» під наявну вибірку). Через це порівняння моделей за звичайним $R^2$ завжди надаватиме перевагу складнішим моделям, тому для адекватного порівняння використовують скоригований $\text{Adj. } R^2$, що містить штраф за кількість параметрів.
4. **Що означає VIF предиктора, що дорівнює приблизно 1, і що означає VIF вище 10 — і що з цим варто зробити на практиці?**
* $VIF \approx 1$ означає повну відсутність лінійного зв'язку цього предиктора з іншими (ідеальтна незалежність).
* $VIF > 10$ свідчить про критичну мультиколінеарність, за якої коефіцієнти стають вкрай нестабільними та ненадійними.
* На практиці від такого предиктора позбуваються (видаляють його з моделі) або застосовують методи зменшення розмірності (наприклад, PCA).



---

## Висновок

Практичну роботу успішно виконано. На кліматичному наборі даних для Києва побудовано моделі множинної лінійної регресії з двома та трьома предикторами, проінтерпретовано коефіцієнти «за інших рівних умов», перевірено мультиколінеарність через VIF, порівняно $R^2$ та скоригований $\text{Adj. } R^2$, а також проведено повну діагностику залишків.